# Pre-Delay Discriminator: Does the 84/84 Contrast Depend on Resonator Photons?

## The tension

Claim 4 ruled out resonator-state-dependent readout for the 84/84 contrast
using an arithmetic bound. But our leading ghost hypothesis (H2) IS
resonator photons causing AC Stark shift. These are in tension:
- If photons cause the 30 kHz ghost, could they also contribute to 84/84?
- The Claim 4 bound assumed photons manifest as readout bias.
  But AC Stark shifts the qubit *frequency*, a different mechanism.

## The test

Insert a variable pre-delay before the 84/84 circuits. If the contrast
Δ=0.59 changes with pre-delay, resonator photons contribute.
If Δ is flat, Claim 4 stands and the ghost is separate from 84/84.


In [ ]:
import torch, math, numpy as np
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
torch.manual_seed(99); np.random.seed(99)


## Model
Resonator photon number decays: nbar(t) = nbar_0 * exp(-t/T1_res).
AC Stark shift: δf = 2χ·nbar(t).
The 84/84 contrast Δ depends on the detuning during the circuit.


In [ ]:
def contrast_vs_predelay(predelay_us, nbar_0=0.01, chi_khz=500, t1_res_us=5.0):
    # Photon number after pre-delay
    nbar = nbar_0 * math.exp(-predelay_us / t1_res_us)
    # AC Stark shift in kHz
    stark_khz = 2 * chi_khz * nbar / 1000.0
    # Model: contrast Δ = 0.59 base, modulated by Stark shift
    # If Stark affects bright/dark differentially, Δ changes
    # Assume differential sensitivity: dΔ/d(stark) = 0.1 per kHz
    delta = 0.59 + 0.1 * stark_khz
    return delta, stark_khz, nbar

print(f'{"Pre-delay (us)":>16} {"nbar":>10} {"Stark (kHz)":>12} {"Δ":>8}')
for pd in [0, 1, 2, 5, 10, 20, 50]:
    d, s, n = contrast_vs_predelay(pd)
    print(f'{pd:>16} {n:>10.4f} {s:>12.3f} {d:>8.3f}')


## Null model: no photon dependence
If Claim 4 is correct and photons don't affect 84/84, Δ is flat.


In [ ]:
def null_contrast(predelay_us):
    return 0.59  # constant

# Simulate measurements with realistic noise
def measure_contrast(predelay_us, model_fn, n_shots=2000):
    true_d = model_fn(predelay_us)
    # Shot noise on the contrast measurement
    se = 0.02  # typical standard error
    return np.random.normal(true_d, se)

predelays = [0, 1, 2, 5, 10, 20, 50]
print('Photonic model measurements:')
for pd in predelays:
    m = measure_contrast(pd, lambda t: contrast_vs_predelay(t)[0])
    print(f'  {pd:>3} us: Δ = {m:.3f}')
print('\nNull model measurements:')
for pd in predelays:
    m = measure_contrast(pd, null_contrast)
    print(f'  {pd:>3} us: Δ = {m:.3f}')


## Discriminator power
Can we distinguish photonic from null with 7 pre-delay points?


In [ ]:
def chi2_photonic_vs_null(n_trials=100):
    detected = 0
    for _ in range(n_trials):
        # Generate data under photonic model
        data = [measure_contrast(pd, lambda t: contrast_vs_predelay(t)[0]) for pd in predelays]
        # Fit null (constant) vs photonic (decaying)
        # Simple: check if slope is significantly negative
        x = np.array(predelays)
        y = np.array(data)
        slope = np.polyfit(x, y, 1)[0]
        # Standard error of slope (rough)
        if slope < -0.0005:  # negative slope = photonic signature
            detected += 1
    return detected / n_trials

power = chi2_photonic_vs_null()
print(f'Detection power: {power:.0%}')
print(f'The pre-delay sweep distinguishes photonic from null: {power > 0.8}')


## Verdict

The pre-delay discriminator resolves the Claim 4 tension:
- **If Δ decays with pre-delay:** resonator photons contribute to 84/84.
  Claim 4's arithmetic bound was too narrow (wrong mechanism).
  The ghost and 84/84 share a physical origin.
- **If Δ is flat:** Claim 4 stands. Photons cause the ghost but not 84/84.
  Two separate phenomena. The 84/84 is purely algebraic.

**For Claude Code:** Add pre-delay sweep [0,1,2,5,10,20,50] µs to the
84/84 ripple circuits. This is the single most informative QPU experiment
for resolving the falsification tension. Costs ~7x the original 84/84
run (already spent, so this is incremental).
